In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression


In [2]:
# list of years for which to produce linear forecasts
future_years = [2027, 2032, 2036, 2046, 2055, 2060]


In [3]:
# creage
dfProjGroupsLinear = pd.DataFrame(
    [
        ["Since 1981", 1981, 2023, {}],
        ["Since 1982", 1982, 2023, {}],
        ["Since 1983", 1983, 2023, {}],
        ["Since 1984", 1984, 2023, {}],
        ["Since 1985", 1985, 2023, {}],
        ["Since 1986", 1986, 2023, {}],
        ["Since 1987", 1987, 2023, {}],
        ["Since 1988", 1988, 2023, {}],
        ["Since 1989", 1989, 2023, {}],
        ["Since 1990", 1990, 2023, {}],
        ["Since 1991", 1991, 2023, {}],
        ["Since 1992", 1992, 2023, {}],
        ["Since 1993", 1993, 2023, {}],
        ["Since 1994", 1994, 2023, {}],
        ["Since 1995", 1995, 2023, {}],
        ["Since 1996", 1996, 2023, {}],
        ["Since 1997", 1997, 2023, {}],
        ["Since 1998", 1998, 2023, {}],
        ["Since 1999", 1999, 2023, {}],
        ["Since 2000", 2000, 2023, {}],
        ["Since 2001", 2001, 2023, {}],
        ["Since 2002", 2002, 2023, {}],
        ["Since 2003", 2003, 2023, {}],
        ["Since 2004", 2004, 2023, {}],
        ["Since 2005", 2005, 2023, {}],
        ["Since 2006", 2006, 2023, {}],
        ["Since 2007", 2007, 2023, {}],
        ["Since 2008", 2008, 2023, {}],
        ["Since 2009", 2009, 2023, {}],
        ["Since 2010", 2010, 2023, {}],
        ["Since 2011", 2011, 2023, {}],
        ["Since 2012", 2012, 2023, {}],
        ["Since 2013", 2013, 2023, {}],
        ["Since 2014", 2014, 2023, {}],
        ["Since 2015", 2015, 2023, {}],
        ["Since 2016", 2016, 2023, {}],
        ["Since 2017", 2017, 2023, {}],
        ["Since 2018", 2018, 2023, {}],
        ["Since 2019", 2019, 2023, {}],
        ["Since 2020", 2020, 2023, {}],
        ["Since 2021", 2021, 2023, {}],
        ["Since 2022", 2022, 2023, {}],
        #    ["Since 2001 w/o 2020", 2001, 2023, {2020}],
        #    ["Since 2011 w/o 2020", 2011, 2023, {2020}]
    ],
    columns=("pgName", "pgYearFrom", "pgYearTo", "pgYearsExclude"),
)

display(dfProjGroupsLinear)


,pgName,pgYearFrom,pgYearTo,pgYearsExclude
0,Since 1981,1981,2023,{}
1,Since 1982,1982,2023,{}
2,Since 1983,1983,2023,{}
3,Since 1984,1984,2023,{}
4,Since 1985,1985,2023,{}
5,Since 1986,1986,2023,{}
6,Since 1987,1987,2023,{}
7,Since 1988,1988,2023,{}
8,Since 1989,1989,2023,{}
9,Since 1990,1990,2023,{}


In [4]:
# import historic AADT (created in previous jupyter notebook)
dfHistoricAadt = pd.read_csv("intermediate/external-historic-aadt.csv")
dfHistoricAadt


,externalid,segid,year,AADT
0,3601,1082_000.0,1981,NaN
1,3601,1082_000.0,1982,NaN
2,3601,1082_000.0,1983,NaN
3,3601,1082_000.0,1984,NaN
4,3601,1082_000.0,1985,NaN
...,...,...,...,...
1242,3629,1826_004.9,2019,3158.0
1243,3629,1826_004.9,2020,3155.0
1244,3629,1826_004.9,2021,3382.0
1245,3629,1826_004.9,2022,3345.0


# Linear forecasts with assist from ChatGPT
https://chat.openai.com/share/d127492a-ad78-4f45-afd0-50e29069db1a

In [5]:
# Initialize a list to store the individual result DataFrames
forecast_results_list = []

# Initialize a set to track which externalids have already been assigned 'No Data'
no_data_externalids = set()

# Open the error file
with open("intermediate/linear-forecasts-errors.txt", "w") as err_file:
    # Loop through the projection groups
    for index, row in dfProjGroupsLinear.iterrows():
        pgName = row["pgName"]
        pgYearFrom = row["pgYearFrom"]
        pgYearTo = row["pgYearTo"]
        pgYearsExclude = set(row["pgYearsExclude"])

        display("Forecasting " + pgName + "...")

        # Group by externalid and segid and iterate through the groups
        for (externalid, segid), group in dfHistoricAadt.groupby(
            ["externalid", "segid"]
        ):
            # Filter the data according to the projection group criteria
            filtered_group = group[
                (group["year"] >= pgYearFrom) & (group["year"] <= pgYearTo)
            ]
            filtered_group = filtered_group[
                ~filtered_group["year"].isin(pgYearsExclude)
            ]

            # Extract X and y
            X = filtered_group["year"].values.reshape(-1, 1)
            y = filtered_group["AADT"].values

            # Filter out NaNs
            valid_mask = ~np.isnan(X.flatten()) & ~np.isnan(y)
            X_valid = X[valid_mask]
            y_valid = y[valid_mask]

            # Check if there is enough data to fit
            if len(X_valid) >= 2:
                model = LinearRegression()
                model.fit(X_valid, y_valid)

                # Predict for the specified future years
                aadt = model.predict(
                    np.array([pgYearFrom] + future_years).reshape(-1, 1)
                )

                # Round the forecasted values to the nearest integers
                aadt = np.rint(aadt).astype(int)

                proj_grp_used = pgName  # Use the real Projection Group Name
            else:
                # Not enough valid data points

                # If externalid already handled, skip
                if externalid in no_data_externalids:
                    continue  # Skip this segid

                # Otherwise, create 'No Data' once for this externalid
                error_msg = f"No valid data for externalid: {externalid}, segid: {segid}, Projection Group: {pgName}. Filling zeros."
                print(error_msg)
                err_file.write(error_msg + "\n")

                # Fill zeros for all years
                aadt = np.zeros(len([pgYearFrom] + future_years), dtype=int)

                proj_grp_used = "No Data"  # Mark as No Data

                # Record that we've handled this externalid
                no_data_externalids.add(externalid)

            # Create a dictionary to store results for this group
            result_dict = {
                "externalid": externalid,
                "segid": segid,
                "PROJ_GRP": proj_grp_used,
            }
            result_dict.update(
                {
                    year: forecast
                    for year, forecast in zip([pgYearFrom] + future_years, aadt)
                }
            )

            # Convert the dictionary to a DataFrame and add to the list
            result_df = pd.DataFrame([result_dict])
            result_df_melt = result_df.melt(
                id_vars=["externalid", "segid", "PROJ_GRP"],
                var_name="year",
                value_name="linear_forecast",
            )
            forecast_results_list.append(result_df_melt)

# Concatenate all the individual result DataFrames
forecast_results = pd.concat(forecast_results_list, ignore_index=True)


'Forecasting Since 1981...'

No valid data for externalid: 3610, segid: 2688_005.5, Projection Group: Since 1981. Filling zeros.
No valid data for externalid: 3619, segid: 3108_000.0, Projection Group: Since 1981. Filling zeros.
No valid data for externalid: 3621, segid: 2865_019.4, Projection Group: Since 1981. Filling zeros.
No valid data for externalid: 3622, segid: 2863_000.0, Projection Group: Since 1981. Filling zeros.
No valid data for externalid: 3625, segid: 2495_000.0, Projection Group: Since 1981. Filling zeros.
No valid data for externalid: 3627, segid: 1822_000.0, Projection Group: Since 1981. Filling zeros.


'Forecasting Since 1982...'

'Forecasting Since 1983...'

'Forecasting Since 1984...'

...

'Forecasting Since 2018...'

'Forecasting Since 2019...'

'Forecasting Since 2020...'

'Forecasting Since 2021...'

'Forecasting Since 2022...'

In [6]:
# check a snippet
forecast_results


,externalid,segid,PROJ_GRP,year,linear_forecast
0,3601,1082_000.0,Since 1981,1981,-26
1,3601,1082_000.0,Since 1981,2027,890
2,3601,1082_000.0,Since 1981,2032,989
3,3601,1082_000.0,Since 1981,2036,1069
4,3601,1082_000.0,Since 1981,2046,1268
...,...,...,...,...,...
6799,3629,1826_004.9,Since 2022,2032,4785
6800,3629,1826_004.9,Since 2022,2036,5361
6801,3629,1826_004.9,Since 2022,2046,6801
6802,3629,1826_004.9,Since 2022,2055,8097


In [7]:
# export csv
forecast_results.to_csv("results/linear-forecasts.csv", index=False)
